In [ ]:
# Importando as bibliotecas
import pandas as pd
import pandera.pandas as pa
import pandera.errors as pa_errors
from data_profiling import ProfileReport
from pathlib import Path

ROOT = Path.cwd().parents[0]
print(ROOT)

DATA = ROOT / "data" / "raw"
REPORTS = ROOT / "reports"


In [ ]:
# carregando as orders
orders = pd.read_csv(
    DATA / "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)

orders.head()

In [ ]:
# Criar um profile do dataset
profile = ProfileReport(
    orders,
    title="Data Profiling - Olist Orders",
    minimal=True,
    explorative=False
)

profile.to_file(REPORTS / "R_01_profile_olist_orders_profile.html")

# ou abrir por: profile.to_notebook_iframe()
# ou gerar html: profile.to_file("olist_orders_profile.html")

In [ ]:
profile

# Data Quality Dimensions


In [ ]:
# ---------------------------------------------------------
# 2. Definição do contrato de qualidade da tabela orders
# ---------------------------------------------------------
orders_schema = pa.DataFrameSchema(
    {

        # -------------------------------------------------
        # UNICIDADE + COMPLETUDE
        # -------------------------------------------------
        # Cada pedido deve possuir um identificador.
        # Como a granularidade desta tabela é:
        #
        # 1 linha = 1 pedido
        #
        # o order_id também deve ser único.
        "order_id": pa.Column(
            str,
            nullable=False,   # não pode ser nulo
            unique=True       # não pode aparecer repetido
        ),


        # -------------------------------------------------
        # COMPLETUDE
        # -------------------------------------------------
        # Todo pedido deve estar associado a um cliente.
        "customer_id": pa.Column(
            str,
            nullable=False
        ),


        # -------------------------------------------------
        # VALIDADE
        # -------------------------------------------------
        # O status do pedido deve pertencer ao conjunto
        # de valores esperados no processo da Olist.
        "order_status": pa.Column(
            str,
            pa.Check.isin([
                "created",
                "approved",
                "invoiced",
                "processing",
                "shipped",
                "delivered",
                "unavailable",
                "canceled",
            ]),
            nullable=False
        ),


        # -------------------------------------------------
        # COMPLETUDE
        # -------------------------------------------------
        # Precisamos saber quando o pedido foi realizado.
        "order_purchase_timestamp": pa.Column(
            "datetime64[ns]",
            nullable=False
        ),


        # -------------------------------------------------
        # COMPLETUDE
        # -------------------------------------------------
        # A aprovação pode estar ausente em alguns casos,
        # por exemplo dependendo do status do pedido.
        #
        # Portanto, inicialmente permitimos NULL.
        #
        # Depois podemos criar uma regra mais específica:
        # "se o pedido foi aprovado/entregue,
        # então order_approved_at deve existir".
        "order_approved_at": pa.Column(
            "datetime64[ns]",
            nullable=True
        ),


        # -------------------------------------------------
        # COMPLETUDE
        # -------------------------------------------------
        # Nem todo pedido necessariamente chegou à transportadora.
        # Pedidos cancelados, por exemplo, podem não possuir essa data.
        "order_delivered_carrier_date": pa.Column(
            "datetime64[ns]",
            nullable=True
        ),


        # -------------------------------------------------
        # COMPLETUDE
        # -------------------------------------------------
        # Nem todo pedido necessariamente foi entregue.
        # Logo, NULL aqui não significa automaticamente erro.
        "order_delivered_customer_date": pa.Column(
            "datetime64[ns]",
            nullable=True
        ),


        # -------------------------------------------------
        # COMPLETUDE
        # -------------------------------------------------
        # O prazo prometido é fundamental para nosso problema,
        # porque será usado para determinar se houve atraso.
        "order_estimated_delivery_date": pa.Column(
            "datetime64[ns]",
            nullable=False
        ),
    },


    # -----------------------------------------------------
    # 3. Regras de CONSISTÊNCIA entre colunas
    # -----------------------------------------------------
    checks=[

        # A aprovação do pagamento não deveria ocorrer
        # antes da criação/compra do pedido.
        #
        # Quando order_approved_at é NULL,
        # não consideramos isso uma violação desta regra.
        pa.Check(
            lambda df:
                df["order_approved_at"].isna()
                |
                (
                    df["order_approved_at"]
                    >= df["order_purchase_timestamp"]
                ),
            error=(
                "order_approved_at não pode ocorrer "
                "antes de order_purchase_timestamp"
            )
        ),

    ]
)


# ---------------------------------------------------------
# 4. Validação
# ---------------------------------------------------------
# lazy=True é útil porque o Pandera tenta encontrar
# várias violações antes de gerar o erro.
#
# Sem lazy=True, ele pode parar logo no primeiro problema.


try:
    validated_orders = orders_schema.validate(
        orders,
        lazy=True
    )

    print("Dados válidos!")

except pa_errors.SchemaErrors as err:
    print("Foram encontrados problemas de qualidade.")
    display(err.failure_cases)


In [ ]:
validated_orders.head()